# M2 Single-Pair Scoring

Notebook equivalent of `python -m src.cli.score`. Choose any M2 generator
from the registry, give it a thumbnail + video vid, get a consistency score.

Use this for: quick spot checks, demos, debugging.


In [ ]:
# [COLAB_BOOTSTRAP_v2_modular]
# Each Colab session:
#   1. Mount Drive             (Drive has the repo + the tar.gz of M1 features)
#   2. cd into the repo        (so relative paths to src/, configs/, etc. work)
#   3. Symlink Drive data dirs (processed/, qvh_frames/, etc.) into the repo
#   4. EXTRACT M1 features tar.gz from Drive ->  LOCAL SSD  (/content/features_local)
#      Reason: 19,000 small .npz files corrupt on Drive's FUSE; LOCAL SSD is fast.
#   5. Clone upstream moment_detr code (for build_model imports)
#
# Drive is ONLY used as the SOURCE of the tar.gz. Features ALWAYS live on
# /content/ (LOCAL SSD) after extraction. Same pattern as
# run_pipeline_allinone.ipynb cell 2.5 and sanity_consistency.ipynb cell 0.

import os, sys, subprocess, time, tarfile, shutil
from pathlib import Path

# ----- 1. Mount Drive ----------------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# ----- 2. Find repo + cd ------------------------------------------------
DRIVE_ROOT = Path('/content/drive/MyDrive/thumbnail_project')
REPO_DIR   = DRIVE_ROOT / 'thumbnail-consistency'
assert REPO_DIR.exists(), f"Upload your repo folder to {REPO_DIR} first."

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# ----- 3. Symlink data dirs ---------------------------------------------
for d in ['processed', 'outputs', 'qvh_videos', 'qvh_frames', 'qvh_thumbnails',
          'qvh_frames_within_neg', 'qvh_frames_multi', 'checkpoints']:
    src, dst = DRIVE_ROOT / d, REPO_DIR / d
    if src.exists() and not dst.exists() and not dst.is_symlink():
        try: os.symlink(src, dst)
        except OSError: pass

# ----- 4. Extract M1 features from Drive .tar.gz -> LOCAL SSD ----------
# Source on Drive (the 7.5 GB archive):
ARCHIVE_ON_DRIVE = DRIVE_ROOT / 'moment_detr_features.tar.gz'
# Destination on LOCAL SSD (NOT on Drive — Drive can't hold 19k small files):
EXTRACTED_LOCAL  = Path('/content/features_local')

def _find_feature_dirs(root):
    """Look inside `root` for clip_features/ and clip_text_features/ dirs
    that contain .npz files. Returns (clip_dir, text_dir) paths or (None, None)."""
    clip = text = None
    for r, dirs, _ in os.walk(root):
        for d in dirs:
            p = Path(r) / d
            if d == 'clip_features' and len(list(p.glob('*.npz'))) > 1000:
                clip = p
            if d == 'clip_text_features' and len(list(p.glob('*.npz'))) > 1000:
                text = p
    return clip, text

# Check ONLY local SSD (NOT Drive) for an already-extracted copy from earlier
# in THIS Colab session. If you restart the runtime, /content is wiped and
# this check returns (None, None), triggering a fresh extraction.
CLIP_FEAT_DIR, TEXT_FEAT_DIR = (None, None)
if EXTRACTED_LOCAL.exists():
    CLIP_FEAT_DIR, TEXT_FEAT_DIR = _find_feature_dirs(EXTRACTED_LOCAL)

if CLIP_FEAT_DIR and TEXT_FEAT_DIR:
    print(f"[features] Found local extraction from earlier this session (skip).")
    print(f"           {EXTRACTED_LOCAL}")
else:
    assert ARCHIVE_ON_DRIVE.exists(), (
        f"Source archive missing on Drive: {ARCHIVE_ON_DRIVE}\n"
        f"This is the only file the bootstrap reads from Drive."
    )
    # Wipe any stale partial extraction and start fresh
    if EXTRACTED_LOCAL.exists():
        shutil.rmtree(EXTRACTED_LOCAL)
    EXTRACTED_LOCAL.mkdir(parents=True)
    sz_gb = ARCHIVE_ON_DRIVE.stat().st_size / 1e9
    print(f"[features] Extracting {sz_gb:.2f} GB:")
    print(f"           FROM (Drive)     : {ARCHIVE_ON_DRIVE}")
    print(f"           TO   (local SSD) : {EXTRACTED_LOCAL}")
    t0 = time.time()
    with tarfile.open(ARCHIVE_ON_DRIVE) as tf:
        tf.extractall(EXTRACTED_LOCAL)
    print(f"[features] Done in {time.time()-t0:.0f}s.")
    CLIP_FEAT_DIR, TEXT_FEAT_DIR = _find_feature_dirs(EXTRACTED_LOCAL)

assert CLIP_FEAT_DIR and TEXT_FEAT_DIR, (
    f"Feature dirs not found after extraction. Check {EXTRACTED_LOCAL}"
)

# ----- 5. Clone upstream Moment-DETR (for build_model imports) ---------
MD_DIR = Path('/content/moment_detr_upstream')
if not MD_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jayleicn/moment_detr.git',
                    str(MD_DIR)], check=True)
if str(MD_DIR) not in sys.path:
    sys.path.insert(0, str(MD_DIR))

# ----- 6. Verification summary -----------------------------------------
import torch
print()
print(f"Repo            : {REPO_DIR}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu (WARN: slow)'}")
print(f"Drive tar.gz    : {ARCHIVE_ON_DRIVE}  ({sz_gb:.2f} GB)" if 'sz_gb' in dir() else "")
print(f"clip_features   : {CLIP_FEAT_DIR}  ({len(list(CLIP_FEAT_DIR.glob('*.npz'))):,} files)")
print(f"clip_text_feats : {TEXT_FEAT_DIR}  ({len(list(TEXT_FEAT_DIR.glob('*.npz'))):,} files)")
print(f"src/ importable : {(REPO_DIR / 'src' / '__init__.py').exists()}")


## 1. Install M2 dependencies (only if you'll use a heavy generator)

- Lightweight M2s (`gt`, `random`, `clipcap`) need **nothing** extra.
- VLMs (`qwen2vl_*`, `llava16_aspect`, `aspect_rewriter`) need this cell.

Skip the next cell if you only plan to run lightweight M2s.


In [ ]:
# Run this if you're using qwen2vl_*, llava16_aspect, or aspect_rewriter.
# Otherwise comment it out — it costs ~1 min on first run.
import sys, subprocess

USE_VLM = True       # ★ set False to skip VLM deps and save 1 min

if USE_VLM:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers>=4.45', 'accelerate>=0.30', 'bitsandbytes>=0.43',
                    'qwen-vl-utils', 'sentencepiece'], check=True)
    print('VLM deps installed.')
else:
    print('Skipped VLM deps (USE_VLM=False).')


## 2. Pick an M2 generator

In [ ]:
from src.m2_generators import build_m2, available
print("Available M2 generators:")
for name in available():
    print(f"  - {name}")

# === EDIT THIS LINE to try a different M2 ===
M2_NAME = 'gt'   # one of: gt, random, clipcap, qwen2vl_simple, qwen2vl_aspect, llava16_aspect

# Per-generator kwargs (some need a lookup or ckpt)
M2_KWARGS = {
    'gt'             : {'lookup_jsonl': 'processed/consistency_eval.jsonl'},
    'random'         : {'pool_jsonl'  : 'processed/consistency_eval.jsonl', 'seed': 2026},
    'clipcap'        : {'ckpt_path'   : 'checkpoints/clipcap_best.pt'},
    'qwen2vl_simple' : {'model_id': 'Qwen/Qwen2-VL-7B-Instruct',          'load_in_4bit': True},
    'qwen2vl_aspect' : {'model_id': 'Qwen/Qwen2-VL-7B-Instruct',          'load_in_4bit': True},
    'llava16_aspect' : {'model_id': 'llava-hf/llava-v1.6-mistral-7b-hf',  'load_in_4bit': True},
}

print(f'\nBuilding M2: {M2_NAME}')
m2 = build_m2(M2_NAME, **M2_KWARGS[M2_NAME])
print(f'Ready: {m2}')


## 2. Build M1 + the consistency scorer (frozen across experiments)

In [ ]:
import yaml
from src.models.moment_detr import MomentDETRFeaturePredictor
from src.scoring.consistency import ConsistencyScorer, ScoringConfig

cfg_yaml = yaml.safe_load(open('configs/consistency.yaml'))

m1 = MomentDETRFeaturePredictor(
    ckpt_path=cfg_yaml['paths']['m1_ckpt'],
    repo_root=str(REPO_DIR),
)
scoring_cfg = ScoringConfig(
    alpha=cfg_yaml['alpha'], tau=cfg_yaml['tau'], top_p=cfg_yaml['top_p'],
    chunk_window=cfg_yaml['chunk_window'], chunk_stride=cfg_yaml['chunk_stride'],
    damp_with_fg=cfg_yaml['damp_with_fg'],
)
# Override feature_dir to the LOCAL extracted path (not the YAML default)
scorer = ConsistencyScorer(m1, str(CLIP_FEAT_DIR), scoring_cfg)
print('Scorer ready.')


## 3. Pick one (thumbnail, video) pair and score it

We pull a random pair from `consistency_eval.jsonl` so the setup is reproducible.
Edit `pair_idx` or set `THUMB_PATH` / `VIDEO_VID` / `QID` manually below.


In [ ]:
import json, random
random.seed(2026)

pairs = [json.loads(l) for l in open('processed/consistency_eval.jsonl', encoding='utf-8') if l.strip()]
pair = random.choice([p for p in pairs if p['label'] == 1])    # take a positive

THUMB_PATH = pair['frame_path']
VIDEO_VID  = pair['video_vid']
QID        = pair['thumb_qid']    # needed only for the 'gt' generator
GT_QUERY   = pair['thumb_query']

print(f'Thumbnail : {THUMB_PATH}')
print(f'Video     : {VIDEO_VID}')
print(f'GT query  : {GT_QUERY}')


## 4. Generate queries from M2 and compute the score

In [ ]:
# Step A: M2 generates K queries from the thumbnail.
#         (For 'gt' we pass qid to look up the human-authored query.)
gen_kwargs = {'qid': QID} if M2_NAME == 'gt' else {}
m2_out = m2.generate_with_metadata(THUMB_PATH, k=3, **gen_kwargs)

print('--- M2 output ---')
print(f'm2_name      : {m2_out["m2_name"]}')
print(f'elapsed_ms   : {m2_out["elapsed_ms"]:.1f}')
for i, q in enumerate(m2_out['queries']):
    print(f'  query[{i}]: {q}')


In [ ]:
# Step B: Plug the queries into the scorer.
result = scorer.score(
    thumbnail_path=THUMB_PATH,
    vid=VIDEO_VID,
    queries=m2_out['queries'],
)

print('--- Consistency score ---')
print(f'C(T, V)      : {result["C"]:+.4f}')
print(f'n_chunks     : {result["n_chunks"]}')
print(f'max_fg_prob  : {result["max_fg_prob"]:.3f}')
print(f'per_query    : {[f"{c:+.3f}" for c in result["per_query"]]}')


## 5. Bundle a result JSON (matches the schema in INTEGRATION_PLAN.md)

In [ ]:
import json

payload = {
    'schema_version': '1.0',
    'consistency_score': float(result['C']),
    'overall_score'    : float(result['C']),
    'm2_generator'     : m2_out['m2_name'],
    'generated_queries': m2_out['queries'],
    'per_query_scores' : [
        {'query': q, 'score': float(c)}
        for q, c in zip(m2_out['queries'], result['per_query'])
    ],
    'temporal'         : {'max_fg_prob': float(result['max_fg_prob']),
                          'n_chunks'   : int(result['n_chunks'])},
    'config'           : {'alpha': scoring_cfg.alpha, 'tau': scoring_cfg.tau,
                          'top_p': scoring_cfg.top_p, 'k_queries': 3},
    'thumbnail'        : {'path': THUMB_PATH},
    'video'            : {'vid': VIDEO_VID},
    'ground_truth'     : {'query': GT_QUERY, 'qid': int(QID)},
    'explanation'      : {
        'm2_metadata' : m2_out.get('metadata', {}),
        'elapsed_ms'  : {'m2_generation': m2_out['elapsed_ms']},
    },
}

print(json.dumps(payload, indent=2, default=str))
